# Olist Data Cleaning & Data Wrangling

## Import pandas

In [2]:
import pandas as pd

## Import files 

In [7]:
orders_df = pd.read_csv('olist_orders_dataset.csv')
order_items_df = pd.read_csv('olist_order_items_dataset.csv')
order_reviews_df = pd.read_csv('olist_order_reviews_dataset.csv')
payments_df = pd.read_csv('olist_order_payments_dataset.csv')
products_df = pd.read_csv('olist_products_dataset.csv')
product_dictionary_df = pd.read_csv('product_category_name_translation.csv')

## Basic exploration of files

### 1. Orders dataset

In [8]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


We can already identify an issue. The data type of the date columns need to be changed to date time for any comparisons to be done correctly. So let's fix that. 

In [6]:
orders_df[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date']] = orders_df[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date']].apply(pd.to_datetime)
orders_df[['order_delivered_customer_date', 'order_estimated_delivery_date']] = orders_df[['order_delivered_customer_date', 'order_estimated_delivery_date']].apply(pd.to_datetime)

orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


## Data types have been changed
![Noice Gif](https://media1.tenor.com/m/H6sjheSkU1wAAAAd/noice-nice.gif)

Lets continue looking at our orders table.

In [9]:
#Get statuses of different orders
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

From here, we can see different order statuses. A large portion of orders have the "delivered" status. Other statuses give us some insight into the dataset and what had happened to other orders. 

What is intriguing is the *"unavailable"* status. What could that possibly mean? We will have to decipher it through some exploration of the data.

### (1.1) Understanding null values

In [10]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

The order null values start at the "order_approved_at" column. Time to examine the statuses of these deliveries and understand the context behind these null values.

#### (i) Order approved at

In [12]:
orders_df.loc[orders_df['order_approved_at'].isnull(), 'order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

The orders which were "canceled" makes sense (it means that the orders were cancelled pre-payment approval), alongside the "created" orders (pressumably means that the order was created but payment has not been approved). 

The puzzling thing here is the *"delivered"* status, because **how can an order be fulfilled, yet payment seems to not have been approved?**. We need to dive deeper. 

In [13]:
#Storing it in a seperate dataframe in case of extra operations on this filtered dataset needs to be done
non_approved_delivered_df = orders_df.loc[(orders_df['order_approved_at'].isnull()) & (orders_df['order_status'] == 'delivered')]
non_approved_delivered_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


The orders did not have a payment approval date, but all the other timestamps have data. The orders were supposedly "fulfilled." We will have to reconcile these orders with the payments table. 

It is a bit unusual but **lets check the next columns.**

#### (ii) Order delivered carrier date

In [14]:
orders_df.loc[orders_df['order_delivered_carrier_date'].isnull(), 'order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

The "unavailable" order status has made an appearance. This means that 609 orders are not visible within the system for some reason. They will have to be dealt with prior to any table joins. 

The number of "cancelled" orders has increased (509). This means the orders were cancelled prior to the order being handed over to the carrier.
The "invoiced" and "processing" orders are interesting. The payments have been approved (the statuses did not show up when the approval date was null), so these statuses are on the end of Olist and/or the carrier. 

The following be can be assumed:

*"Processing" orders - Orders that are being packaged and will make way to the carrier.*

*"Invoiced" orders - Orders where the package is **either** en route to the carrier or yet to be processed.*

The "delivered" orders are puzzling. The value decreased from the previous column. Let's do some investigation. 

In [24]:
orders_df.loc[(orders_df['order_delivered_carrier_date'].isnull()) & (orders_df['order_status'] == 'delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


Looking at this, we see:

(i) There 2 orders, both have been approved. 

(ii) One order **has a customer delivery date, but no carrier delivery date**, and the other **has no customer or carrier delivery date**.

The former order will have to be checked with the payments table, and the latter will probably be removed in any operations for sales and revenue.

In the other columns

The new order statuses have reached the total count of their status (Unavailable = 609; Created = 5; Approved = 2). We'll have to monitor these values for any changes.

*I'm aware that we'll probably remove most of these rows (for fulfillment and revenue analysis) once tables are joined, however, I need to determine the status of orders for analysis of operations pertaining to logistics*

#### (iii) Order delivered customer date

In [19]:
orders_df.loc[orders_df['order_delivered_customer_date'].isnull(), 'order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

New order status "shipped" indicates to us that orders have reached the carrier but are yet to be delivered. 

The following statuses have not changed values: unavailable, invoiced, processing, created, approved.

The number of "cancelled" shipments have increased - meaning they were cancelled after arriving at the carrier, but before being delivered. 

"Delivered" orders have changed values once more. At this point, I must change my career to Detective, with all of the investigations happening.

In [25]:
orders_df.loc[(orders_df['order_delivered_customer_date'].isnull()) & (orders_df['order_status'] == 'delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


Looking at this, we see:

(i) There 8 orders. 

(ii) 7 orders **have a carrier delivery date, but no customer delivery date**, and 1 order **has no customer or carrier delivery date**.

Same process, check payments table and remove a record.

**The last column has no null values**

### (1.2) Checking for duplicates

We have to verify that the data has no duplicates. We'll be using the "order_id" column for verification. It is the *only* truly unique value in each row.

In [29]:
orders_df.duplicated('order_id').value_counts()

False    99441
Name: count, dtype: int64

No duplicate rows in the table. That's good :)

### (1.3) Ensuring data integrity

Now that we have checked for duplicates and understood the context behind the null values, we need to verify that the data is trustworthy in terms of data validity and integrity. 

We cannot assume that all the dates of the table align. It is enitrely possible that "delivery" dates are **somehow before the date of purchase or approval**. We need to check to ensure that the data is valid.

#### (i) Order purchase vs other columns

In [43]:
print('Comparison (purch vs approve): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_approved_at']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_carrier): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_carrier_date']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_customer): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_customer_date']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_estimate): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_estimated_delivery_date']].shape[0], '# of incorrect rows')

Comparison (purch vs approve):  0 # of incorrect rows
Comparison (purch vs deliv_carrier):  166 # of incorrect rows
Comparison (purch vs deliv_customer):  0 # of incorrect rows
Comparison (purch vs deliv_estimate):  0 # of incorrect rows


There are 166 rows of purchase dates being later than the date of delivery to the carrier. Let's see the data visually.

In [46]:
orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_carrier_date']].head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04 00:00:00
1111,ad133696906f6a78826daa0911b7daec,e6f5b234bb0d847f10eebd70130c5d49,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18 00:00:00
1329,74e033208dc13a7b8127eb8e73d09b76,72fcbb1145f2889eddcba6d6c1d6c090,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29 00:00:00
1372,a6b58794fd2ba533359a76c08df576e3,ccd8e3459ad58ae538c9d2cf35532ba4,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08 00:00:00
1864,5792e0b1c8c8a2bf53af468c9a422c88,5c427f06e7fcf902e309743516e2c580,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09 00:00:00
2760,c3eb293fd154223498b6551a728203e8,1701ed88b1e8cb21051660d70d11d038,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06 00:00:00
3473,b0c2a7d04b165525254254a728c50a4e,76a8a3092e48bba0826495241903e450,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04 00:00:00
3661,2033a4586b5bec3229ebc1675a8ae092,0efc1f5e8675901683fae0ea8fdf3feb,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19 00:00:00
4114,08adcddad19d3acf37d1fa01cb9ded1e,94c2f45091985c47b4c4fb301113f0a4,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18 00:00:00
4159,dee6298ce7d1fb2645141ef9972157aa,c35fd04a09c83d6f53b2d4557eab7445,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaN,2018-05-28 00:00:00


Upon closer inspection, we see that the dates align but the times of the carrier delivery date compared to the purchase time are off.

Let's check other columns.

#### (ii) Order payment approval vs other columns

In [44]:
print('Comparison (approve vs deliv_carrier): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_carrier_date']].shape[0], '# of incorrect rows')
print('Comparison (approve vs deliv_customer): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_customer_date']].shape[0], '# of incorrect rows')
print('Comparison (approve vs deliv_estimate): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_estimated_delivery_date']].shape[0], '# of incorrect rows')

Comparison (approve vs deliv_carrier):  1359 # of incorrect rows
Comparison (approve vs deliv_customer):  61 # of incorrect rows
Comparison (approve vs deliv_estimate):  12 # of incorrect rows


In [47]:
orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_carrier_date']].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00


In [ ]:
orders_df.sort_values('order_purchase_timestamp').head(15)

### 2. Order Items dataset

### 3. Order Reviews dataset

### 4. Order Paymnets dataset

### 5. Products dataset

## Dataframe manipulation

## Combining datasets

## Cleaning dataframe & engineering features